# 数据、特征与自行训练

本 Notebook 展示项目的基本数据处理和训练入口。将比赛的三个 CSV 放入 `data/` 后可运行。
历史最佳自行训练公开成绩为 **0.80593**；早期随机分层结果与后来默认使用的同行分组验证需区分。
完整实验过程见 `docs/PROJECT_SUMMARY.md`。

In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p/'spaceship/train.py').exists())
sys.path.insert(0,str(ROOT))

## 检查数据结构与同行组

In [2]:
train=pd.read_csv(ROOT/'data/train.csv',dtype={'PassengerId':str})
test=pd.read_csv(ROOT/'data/test.csv',dtype={'PassengerId':str})
assert train.PassengerId.is_unique and test.PassengerId.is_unique
assert 'Transported' in train and 'Transported' not in test
train_groups=set(train.PassengerId.str.split('_').str[0])
test_groups=set(test.PassengerId.str.split('_').str[0])
print(f'训练集 {len(train):,} 人；测试集 {len(test):,} 人。')
print('训练与测试的同行组交集：',len(train_groups&test_groups))
display(train.isna().sum().to_frame('缺失数'))

训练集 8,693 人；测试集 4,277 人。
训练与测试的同行组交集： 0


,缺失数
PassengerId,0
HomePlanet,201
CryoSleep,217
Cabin,199
Destination,182
Age,179
VIP,203
RoomService,181
FoodCourt,183
ShoppingMall,208


## 特征如何产生
缺失消费不直接等于0。全缺失消费合计仍保留NaN，类别缺失表示Unknown。
增强特征使用公开名册计算同行人数、同姓人数等，不读取测试答案；同姓人数不是确定的家庭人数。

In [3]:
from spaceship.features import build_features
roster=pd.concat([train.drop(columns='Transported'),test],ignore_index=True)
base=build_features(train)
full=build_features(train,roster,enhanced=True)
print('基础特征数：',base.shape[1],'；增强特征数：',full.shape[1])
print('新增字段：',', '.join(c for c in full if c not in base))

基础特征数： 15 ；增强特征数： 30
新增字段： GroupSize, IsAlone, GroupNumber, GroupPosition, SurnameCount, CabinRegion, SpendState, LuxurySpend, AmenitiesUsed, AgeMissing, RoomServiceMissing, FoodCourtMissing, ShoppingMallMissing, SpaMissing, VRDeckMissing


## 分组验证与模型训练
发布版默认按同行组划分五折，训练与验证不共享同行组。数值填补只在训练折拟合；CatBoost原生处理数值NaN。
设 `RUN_TRAINING=True` 后会训练新的模型并保存至 `runs/notebook-training/`；不会修改比赛原始CSV。
阈值固定0.5。以下默认先阅读结果，不重复训练。

In [4]:
RUN_TRAINING=False
if RUN_TRAINING:
    result=subprocess.run([sys.executable,'-m','spaceship.train','--model','catboost',
        '--features','full','--split','group','--output-dir','runs/notebook-training'],
        cwd=ROOT,check=True,capture_output=True,text=True)
    print(result.stdout)
else:
    print('需要训练时，将 RUN_TRAINING 改为 True 后执行本格。')

需要训练时，将 RUN_TRAINING 改为 True 后执行本格。


## 历史实际提交结果
这里只记录项目账号已收到的公开榜分数。第四轮为公开概率复现，其成绩归因与自行训练方案不同。

In [5]:
display(pd.read_csv(ROOT/'reports/public_scores.csv'))

,stage,method,public_accuracy,origin,evidence
0,v1,Random forest,0.80032,locally_trained,user_reported_submission_result
1,v2,CatBoost with engineered features,0.80593,locally_trained,user_reported_submission_result
2,v3,Retained v2 after unsuccessful experiments,0.80593,identical_to_v2,user_reported_unchanged_result
3,v4_baseline,Four public TabPFN probabilities with logistic...,0.82604,public_probability_replay,user_provided_Kaggle_submission_screenshot_202...
4,v4_blend,Public TabPFN and CatBoost stack blend,0.82534,public_probability_replay,user_provided_Kaggle_submission_screenshot_202...


## 后续阅读
参见 `spaceship/train.py`、`spaceship/features.py` 和 `spaceship/ablation.py`。
例如使用 `--features no_surname` 独立移除同姓人数；使用 `--split stratified` 复查早期划分，但不把两种验证的分数直接比较。